In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [2]:
#load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
#load finetuned model
finetuned_model_path = "./final_model"
ft_model = PeftModel.from_pretrained(base_model, finetuned_model_path)

ft_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [4]:
prompt = "Generate a medium difficulty dungeon with 6 rooms"

In [5]:
messages = [{"role": "user", "content":prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

In [6]:
#device management
device = next(ft_model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

In [7]:
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=16384,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=None,
        repetition_penalty=1.0
    )

In [11]:
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
generated_content = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(generated_content)

<think>

</think>

{"tiles": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
import re
import json
import os

def extract_and_complete_json(text):
    """Extract JSON and complete it if it's just missing closing braces."""
    # Remove reasoning tags
    text = re.sub(r'`<think>`.*?`</think>`', '', text, flags=re.DOTALL)
    text = text.strip()
    
    json_start = text.find('{')
    if json_start == -1:
        return None
    
    json_part = text[json_start:]
    
    # Count braces
    open_braces = json_part.count('{')
    close_braces = json_part.count('}')
    missing = open_braces - close_braces
    
    print(f"Open braces: {open_braces}, Close braces: {close_braces}, Missing: {missing}")
    
    if missing == 0:
        # Complete - try to parse
        try:
            return json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"JSON parse error: {e}")
            return None
    elif missing == 1:
        # Missing one closing brace - try to complete it
        print("Attempting to complete JSON by adding missing closing brace...")
        
        # Check what fields we have
        has_tiles = '"tiles"' in json_part
        has_player = '"player_spawn"' in json_part
        has_stairs = '"stairs_spawn"' in json_part
        has_width = '"width"' in json_part
        has_height = '"height"' in json_part
        
        print(f"Has tiles: {has_tiles}, player_spawn: {has_player}, stairs_spawn: {has_stairs}, width: {has_width}, height: {has_height}")
        
        # Remove trailing whitespace and comma
        json_part = json_part.rstrip()
        if json_part.endswith(','):
            json_part = json_part[:-1].rstrip()
        
        # Count brackets
        open_brackets = json_part.count('[')
        close_brackets = json_part.count(']')
        missing_brackets = open_brackets - close_brackets
        
        print(f"Open brackets: {open_brackets}, Close brackets: {close_brackets}, Missing: {missing_brackets}")
        
        # Close any missing array brackets first
        for _ in range(missing_brackets):
            json_part += ']'
        
        # Now add missing fields if needed
        if not has_player or not has_stairs or not has_width or not has_height:
            # IMPORTANT: Add comma after closing the tiles array
            json_part += ','
            
            if not has_player:
                json_part += ' "player_spawn": [0, 0]'
            if not has_stairs:
                json_part += ', "stairs_spawn": [0, 0]'
            if not has_width:
                json_part += ', "width": 56'
            if not has_height:
                json_part += ', "height": 32'
            if '"difficulty"' not in json_part:
                json_part += ', "difficulty": "medium"'
            if '"enemies"' not in json_part:
                json_part += ', "enemies": []'
        
        # Finally, close the object
        json_part += '}'
        
        try:
            return json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"Failed to complete JSON: {e}")
            print(f"Error at position: {e.pos}")
            print(f"Attempted completion (last 300 chars): {json_part[-300:]}")
            # Try to show context around the error
            error_start = max(0, e.pos - 50)
            error_end = min(len(json_part), e.pos + 50)
            print(f"Error context: ...{json_part[error_start:error_end]}...")
            return None
    else:
        print(f"⚠ JSON missing {missing} closing braces - too incomplete to fix")
        return None

# Use it
map_json = extract_and_complete_json(generated_content)

if map_json:
    print("✓ Successfully extracted and completed JSON!")
    print(f"Keys: {list(map_json.keys())}")
    
    # Validate required fields
    required = ['tiles', 'player_spawn', 'stairs_spawn', 'width', 'height']
    missing = [key for key in required if key not in map_json]
    if missing:
        print(f"⚠ Missing required fields: {missing}")
    else:
        print("✓ All required fields present")
        
        # Check tiles dimensions
        if 'tiles' in map_json:
            tiles = map_json['tiles']
            print(f"Tiles shape: {len(tiles)} rows x {len(tiles[0]) if tiles else 0} cols")
        
        # Create directory if it doesn't exist
        map_id = "map_generated_001"
        maps_dir = "../web_game/maps"
        os.makedirs(maps_dir, exist_ok=True)
        
        map_path = f"{maps_dir}/{map_id}.json"
        with open(map_path, 'w') as f:
            json.dump(map_json, f, indent=2)
        print(f"✓ Saved to {map_path}")
        
        # Also update the map_index.json
        index_path = f"{maps_dir}/map_index.json"
        if os.path.exists(index_path):
            with open(index_path, 'r') as f:
                index = json.load(f)
        else:
            index = {"total_maps": 0, "map_ids": [], "source": "generated", "split": "generated"}
        
        if map_id not in index["map_ids"]:
            index["map_ids"].append(map_id)
            index["total_maps"] = len(index["map_ids"])
            with open(index_path, 'w') as f:
                json.dump(index, f, indent=2)
            print(f"✓ Added {map_id} to map_index.json")
else:
    print("✗ Could not extract or complete JSON")

Open braces: 1, Close braces: 0, Missing: 1
Attempting to complete JSON by adding missing closing brace...
Has tiles: True, player_spawn: False, stairs_spawn: False, width: False, height: False
Open brackets: 99, Close brackets: 97, Missing: 2
✓ Successfully extracted and completed JSON!
Keys: ['tiles', 'player_spawn', 'stairs_spawn', 'width', 'height', 'difficulty', 'enemies']
✓ All required fields present
Tiles shape: 98 rows x 56 cols
✓ Saved to ../web_game/maps/map_generated_001.json
✓ Added map_generated_001 to map_index.json


In [18]:
index_file = "../web_game/maps/map_index.json"
with open(index_file, 'r') as f:
    index = json.load(f)

if map_id not in index["map_ids"]:
    index["map_ids"].append(map_id)
    index["total_maps"] = len(index["map_ids"])

    with open(index_file, 'w') as f:
        json.dump(index, f, indent=2)
    print(f"Added {map_id} to index")